In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [15]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)  
    add_assistant_message(messages, " ''''json")  
    response = chat(messages, stop_sequences=["''''"])

    return json.loads(response)

In [16]:
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that parses an AWS S3 bucket name from an S3 URI (e.g., 's3://my-bucket/path/to/file') and returns just the bucket name."}, {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific DynamoDB table named 'UserProfiles'."}, {'task': 'Write a regular expression that matches valid AWS EC2 security group IDs (format: sg- followed by 8 or 17 hexadecimal characters).'}]


In [17]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [18]:
# This is the test case to merge the prompt with generated dataset testcases.
def run_promt(testcase):
    prompt = f""" 
Please solve the following testcase: {testcase['task']}
"""
    messages = []
    add_user_message(messages, prompt)  
    response = chat(messages)

    return response

In [19]:
# This function acts as orchestrator to run the prompt evaluation testcases.
def run_test_cases(test_case):
    ''' Calls run_promt and grades the response based on correctness and conciseness. ''' 
    output = run_promt(test_case)
    # Grade the output based on correctness and conciseness.
    score = 10
    
    graded_output = {
        "output": output,
        "test_case": test_case,
        "score": score
    }
    return graded_output


In [20]:
# This is the eval pipeline that runs the test cases and collects the results.
def eval_pipeline(dataset):
    results = []
    for test_case in dataset:
        result = run_test_cases(test_case)
        results.append(result)
    
    return results

In [21]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
    
results = eval_pipeline(dataset)


In [22]:
print( json.dumps(results, indent=2) )

[
  {
    "output": "# AWS S3 Bucket Name Parser\n\nHere's a Python function that parses the bucket name from an S3 URI:\n\n```python\ndef parse_s3_bucket_name(s3_uri: str) -> str:\n    \"\"\"\n    Parse the bucket name from an S3 URI.\n    \n    Args:\n        s3_uri: S3 URI in the format 's3://bucket-name/path/to/file'\n        \n    Returns:\n        The bucket name as a string\n        \n    Raises:\n        ValueError: If the URI is not a valid S3 URI\n    \"\"\"\n    # Validate and normalize the input\n    if not isinstance(s3_uri, str):\n        raise ValueError(\"S3 URI must be a string\")\n    \n    s3_uri = s3_uri.strip()\n    \n    # Check for s3:// prefix\n    if not s3_uri.startswith('s3://'):\n        raise ValueError(\"S3 URI must start with 's3://'\")\n    \n    # Remove the s3:// prefix\n    uri_without_prefix = s3_uri[5:]\n    \n    # Check if URI is empty after removing prefix\n    if not uri_without_prefix:\n        raise ValueError(\"S3 URI must contain a bucket na